In [ ]:
%pip install -q "math-verify[antlr4_13_2]==0.9.0" torch \
  "transformers==5.0.0" "trl==1.10.0" "peft==0.19.1" \
  accelerate datasets "torchao>=0.16.0"

import subprocess, os, sys, base64, io, tarfile
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "/kaggle/working/DACNTT_2026"])
if subprocess.run(["git", "clone", "https://github.com/tsan2711/DACNTT_2026.git", "/kaggle/working/DACNTT_2026"]).returncode != 0:
    raise SystemExit("git clone thất bại")
os.chdir("/kaggle/working/DACNTT_2026")
BLOB = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall(".")
env = os.environ.copy()
env["PYTHONPATH"] = "src:."
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
heartbeat = subprocess.Popen(
    [sys.executable, "-u", "experiments/viec3/heartbeat.py", "/kaggle/working/heartbeat.log"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL, start_new_session=True)
print("setup xong")


In [ ]:
import subprocess, sys, os, json
os.environ.update(env)
sys.path.insert(0, "experiments/viec3")
from kaggle_driver import run_child, StallTracker
out = "/kaggle/working/results/viec3/teacher-gen-7b"
os.makedirs(out, exist_ok=True)
cmd = ["python", "-u", "-m", "experiments.viec3.teacher_gen", "--model", "Qwen/Qwen2.5-7B-Instruct", "--out", out,
       "--seed", "1", "--k", "4", "--max-tokens", "1024", "--keep", "298"]
for attempt in range(4):                       # resumable: a stalled/killed attempt continues where it stopped
    status = run_child(cmd, __import__("pathlib").Path("/kaggle/working/run_teacher-gen-7b.log"), StallTracker(30 * 60), poll=10.0)
    print("attempt", attempt, status, flush=True)
    if os.path.exists(out + "/summary.json"):
        break
print(open(out + "/summary.json").read() if os.path.exists(out + "/summary.json") else "NO SUMMARY")
